# Dashboard: Live Status Monitor

**Purpose:** Subscribe to all MQTT topics and display real-time aggregated status.

**Subscribes to:** All topics under `city/flood/#`

**Display:** Sensor count, average water level, current alert status

**Update Interval:** Every 2 seconds

In [ ]:
import time
import json
from datetime import datetime, timezone
from pathlib import Path
import sys
from collections import defaultdict

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import ObserverReading, ControlCommand

# Load configuration
try:
    cfg = load_config()
    print(f"Dashboard configured for Køge flood simulation")
    print(f"  Map center: Køge ({cfg.mqtt.base_topic})")
except Exception as e:
    print(f"Error loading config: {e}")
    cfg = None

# Data storage
sensor_readings = {}  # sensor_id -> (water_level, timestamp)
control_status = {"alert": "unknown", "timestamp": None}
update_count = 0

In [ ]:
if cfg:
    mqtt = MqttConnector(cfg.mqtt, client_id_suffix='dashboard')
    mqtt.connect()
    if mqtt.wait_for_connection(timeout=5):
        print("✓ Connected to MQTT broker")
    else:
        print("✗ Failed to connect to MQTT broker")
        mqtt = None


def on_message(client, userdata, msg):
    """Generic message handler for all topics."""
    global sensor_readings, control_status
    
    try:
        payload = msg.payload.decode()
        topic = msg.topic
        
        # Parse observer readings
        if "observer/sensor_" in topic:
            data = json.loads(payload)
            reading = ObserverReading.from_json_dict(data)
            sensor_readings[reading.sensor_id] = (reading.water_level, reading.timestamp)
        
        # Parse control commands
        elif "control/command" in topic:
            data = json.loads(payload)
            cmd = ControlCommand.from_json_dict(data)
            control_status["alert"] = cmd.target
            control_status["timestamp"] = cmd.timestamp
            
    except Exception as e:
        pass  # Silently ignore parse errors


# Subscribe to all topics
if mqtt:
    base_topic = make_topic(cfg.mqtt, "#")
    mqtt.client.subscribe(base_topic, qos=1)
    mqtt.client.on_message = on_message
    print(f"✓ Subscribed to all topics under: {cfg.mqtt.base_topic}/#")

In [ ]:
def display_status():
    """Display aggregated status with Phase 5 enhancements (real distances)."""
    global update_count
    
    update_count += 1
    
    # Calculate average water level
    if sensor_readings:
        avg_level = sum(level for level, _ in sensor_readings.values()) / len(sensor_readings)
        sensor_count = len(sensor_readings)
    else:
        avg_level = 0
        sensor_count = 0
    
    # Determine alert indicator
    alert_status = control_status["alert"]
    if alert_status == "evacuation":
        indicator = "🔴 HIGH"
    elif alert_status == "return":
        indicator = "🟢 LOW"
    else:
        indicator = "⚪ UNKNOWN"
    
    # Phase 5: Calculate real distance between evacuation zones
    from simulated_city.geo import distance_wgs84
    
    torv_lat, torv_lon = 55.4566, 12.1818  # Køge Torv (safe)
    strand_lat, strand_lon = 55.4506, 12.1975  # Køge Søndre Strand (danger)
    
    evac_distance = distance_wgs84(torv_lat, torv_lon, strand_lat, strand_lon)
    evac_distance_km = evac_distance / 1000
    
    # Print status with enhanced info
    print(f"[{update_count}] SENSORS: {sensor_count} active  |  Avg water: {avg_level:.2f}m")
    print(f"         ALERT: {indicator}  |  Evac distance: {evac_distance_km:.2f}km")

In [ ]:
print("\n" + "="*60)
print("LIVE STATUS - Køge Flood Simulation")
print("="*60)
print("(Press Ctrl+C to stop)\n")

try:
    while True:
        display_status()
        time.sleep(2)  # Update every 2 seconds
        
except KeyboardInterrupt:
    print("\n\n✓ Dashboard stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()

In [ ]:
print("\n" + "="*60)
print("LIVE STATUS - Køge Flood Simulation")
print("="*60)
print("(Press Ctrl+C to stop)\n")

update_count = 0
try:
    while True:
        update_count += 1
        
        with data_lock:
            # Display sensor readings
            if sensor_data:
                water_levels = [s["water_level"] for s in sensor_data.values()]
                avg_level = sum(water_levels) / len(water_levels) if water_levels else 0
                print(f"[{update_count}] SENSORS: {len(sensor_data)} active  |  Avg water: {avg_level:.2f}m")
            else:
                print(f"[{update_count}] SENSORS: waiting for data...")
            
            # Display alert status
            alert_level = alert_status["level"]
            alert_icon = "🔴" if alert_level == "high" else "🟢"
            print(f"         ALERT: {alert_icon} {alert_level.upper()}")
        
        time.sleep(2)

except KeyboardInterrupt:
    print("\n\nDashboard stopped.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

## Live Status Display

Real-time stats from MQTT streams (auto-updates while running)

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="dashboard")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

def on_message(client, userdata, msg):
    """Universal callback to process all incoming messages."""
    try:
        data = json.loads(msg.payload.decode())
        topic = msg.topic
        
        with data_lock:
            if "/observer/" in topic:
                # Water level reading
                sensor_id = data.get("sensor_id", "unknown")
                sensor_data[sensor_id] = {
                    "water_level": data.get("water_level", 0.0),
                    "timestamp": data.get("timestamp", "")
                }
            elif "/control/command" in topic:
                # Control command (alert status)
                severity = data.get("parameters", {}).get("severity", "low")
                alert_status["level"] = severity
                alert_status["timestamp"] = data.get("timestamp", "")
    except Exception as e:
        pass  # Silently ignore parse errors

connector.client.on_message = on_message
base_topic = mqtt_cfg.base_topic
connector.client.subscribe(f"{base_topic}/#")

print(f"✓ Subscribed to all topics under: {base_topic}/#")

## Connect to MQTT and Subscribe to All Streams

In [ ]:
import time
import json
import threading
from datetime import datetime, timezone
from collections import defaultdict

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt

# Locations
KOEGE_TORCH_LAT = 55.456553861769855
KOEGE_TORCH_LON = 12.181774944848945
KOEGE_STRAND_LAT = 55.45058843620187
KOEGE_STRAND_LON = 12.197503222443261

# Data storage
sensor_data = {}  # sensor_id -> {"water_level": float, "timestamp": str}
pedestrians = {}  # person_id -> {"lat": float, "lon": float, "location": str}
alert_status = {"level": "low", "timestamp": None}
data_lock = threading.Lock()

print("Dashboard configured for Køge flood simulation")
print(f"  Map center: Køge ({KOEGE_STRAND_LAT:.4f}, {KOEGE_STRAND_LON:.4f})")

## Setup and Configuration

# Dashboard: Live Flood Simulation Monitor

**Real-time visualization of the flood simulation using anymap-ts.**

Displays:
- **Sensors**: Blue markers at Køge Søndre Strand with water level
- **Pedestrians**: Green markers at current position (strand or torv)
- **Alert Status**: Color-coded (green = safe, red = danger)
- **Water Level**: Displayed in status bar

All data streams from MQTT brokers in real-time.

# Dashboard Notebook
Subscribes to all MQTT topics and renders a live map using `anymap-ts`.
Used for visualizing the flood simulation.